# Week 29, Day 5: Ion channels and pumps: counting channels and paying for gradients

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Channels per square micrometre

Write `density(g_max, gamma)` giving channels per μm² from a maximal conductance in mS/cm² and a unitary conductance in pS (1 cm² is 10⁸ μm²). Use it for Hodgkin and Huxley's sodium (120 mS/cm², 4 pS) and potassium (36 mS/cm², take 15 pS) conductances.

In [ ]:
import numpy as np

def density(g_max, gamma):
    return ...

dens = {"Na": density(120, 4), "K": density(36, 15)}
for ion, n in dens.items():
    print(f"{ion}: {n:.0f} channels per um^2")
area_soma = np.pi * 20 ** 2                    # um^2, a 20 um sphere
print(f"a 20 um soma at the squid sodium density would hold {dens['Na'] * area_soma:.2g} sodium channels")

In [ ]:
import numpy as np
def test_density():
    "density converts mS/cm^2 to S/um^2 and divides by the unitary conductance"
    assert abs(density(120, 4) - 120e-3 / 1e8 / 4e-12) < 1e-9 and abs(density(120, 4) - 300) < 1e-9

def test_values():
    "dens holds the sodium and potassium densities, about 300 and 18 to 36 per um^2"
    assert abs(dens["Na"] - 300) < 1e-9 and 15 < dens["K"] < 40

test_density()
test_values()
print("All checks passed")

### Exercise: Non-stationary noise analysis

Two hundred sweeps of a current through 500 channels of 1 pA each are simulated, with 0.5 pA of background noise. Compute the mean `m` and variance `v` across sweeps at each time (subtract the background variance 0.25), then fit v = i m - m²/N by linear least squares on the two columns m and -m², and recover `i_est` and `N_est`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(5)
N, i, sweeps = 500, 1.0, 200
t = np.arange(0, 10, 0.05)                                  # ms
p = np.exp(-t / 3) - np.exp(-t / 0.5); p = 0.6 * p / p.max()  # open probability: rises, then inactivates
I = i * r.binomial(N, p, size=(sweeps, t.size)) + r.normal(0, 0.5, (sweeps, t.size))

m = ...
v = ...
X = np.column_stack([m, -m ** 2])
coef = ...
i_est, N_est = coef[0], 1 / coef[1]
print(f"unitary current {i_est:.2f} pA, channels {N_est:.0f}")
mm = np.linspace(0, m.max(), 100)
plt.plot(m, v, ".", label="data"); plt.plot(mm, i_est * mm - mm ** 2 / N_est, label="fit")
plt.xlabel("mean current (pA)"); plt.ylabel("variance (pA^2)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_moments():
    "m and v are the across-sweep mean and the variance corrected for 0.5 pA of background noise"
    assert np.allclose(m, I.mean(0)) and np.allclose(v, I.var(0, ddof=1) - 0.25)

def test_fit():
    "the fit recovers the unitary current within 10% and the channel number within 15%"
    assert abs(i_est - 1.0) < 0.1 and abs(N_est - 500) / 500 < 0.15

test_moments()
test_fit()
print("All checks passed")

### Exercise: Paying for the resting leak

A neuron with input resistance 200 MΩ has a resting sodium leak of 20 pA. Compute the sodium ions entering per second `na_per_s`, the pump cycles per second needed to remove them `cycles`, the pump current `I_pump` (A), the hyperpolarisation it causes `dV_pump` (mV), and the power `power` (W) at 8e-20 J per ATP. Then scale the power to 86 billion neurons.

In [ ]:
import numpy as np
e, I_Na, R_in, e_atp = 1.602e-19, 20e-12, 200e6, 8e-20
na_per_s = ...
cycles = ...
I_pump = ...
dV_pump = ...
power = ...
print(f"{na_per_s:.2e} Na+ per second, {cycles:.2e} pump cycles and ATP per second")
print(f"pump current {I_pump * 1e12:.1f} pA, hyperpolarising by {dV_pump:.1f} mV")
print(f"power {power:.2e} W per neuron, {power * 86e9:.2f} W for 86 billion")

In [ ]:
import numpy as np
def test_rates():
    "na_per_s is the sodium current divided by e, and cycles is a third of that"
    assert abs(na_per_s - 20e-12 / 1.602e-19) < 1 and abs(cycles - na_per_s / 3) < 1

def test_costs():
    "the pump current is one charge per cycle, its voltage effect is I_pump R_in, and the power is cycles times 8e-20 J"
    assert abs(I_pump - cycles * 1.602e-19) < 1e-20 and abs(dV_pump - I_pump * 200e6 * 1e3) < 1e-9 and abs(power - cycles * 8e-20) < 1e-24

test_rates()
test_costs()
print("All checks passed")

### Exercise: Channel noise in thin axons

For axons of diameter 0.1, 0.2, 0.5 and 1 μm, count the sodium channels `n_chan` in a 10 μm length at 60 per μm², and compute the relative fluctuation `cv` of the number open when each channel is open with probability p_open = 0.02, as it might be near rest.

In [ ]:
import numpy as np
d = np.array([0.1, 0.2, 0.5, 1.0])        # um
p_open = 0.02
n_chan = ...
cv = ...
for di, n, c in zip(d, n_chan, cv):
    print(f"d = {di:.1f} um: {n:5.0f} channels, {n * p_open:5.1f} open on average, relative fluctuation {c:.2f}")

In [ ]:
import numpy as np
def test_counts():
    "n_chan is pi d L times the density, for a 10 um length"
    assert np.allclose(n_chan, np.pi * d * 10 * 60)

def test_cv():
    "cv is sqrt((1 - p) / (N p)) and exceeds 0.4 for the 0.1 um axon"
    assert np.allclose(cv, np.sqrt((1 - p_open) / (n_chan * p_open))) and cv[0] > 0.4

test_counts()
test_cv()
print("All checks passed")